# 📱 Ajay AI Assistant — Deploy (Bina Computer Ke)

---

## 🔴 Pehle ye padho — RED hamesha error nahi hota!

Colab me **do tarah ka red** hota hai. Farq samajhna zaroori hai:

| Kya dikh raha hai | Matlab | Karna kya hai |
|---|---|---|
| **Gulabi/red background** me normal text (jaise `WARNING`, `notice`) | Ye sirf **message** hai, error nahi | **Kuch nahi** — aage badho ✅ |
| **`---------`** wali lambi line + `Error` / `Traceback` | Ye **asli error** hai | Neeche 🩺 wala cell chalao |
| Cell ke left me **लाल ⊗** nishaan | Cell fail hua | Neeche 🩺 wala cell chalao |

> `pip install` aur `pytest` apne messages **stderr** pe bhejte hain, aur
> Colab stderr ko **gulabi background** me dikhata hai. Wo bilkul normal hai.

---

### Chalane ka tareeka
Har cell ke left me **▶** button hai. **Upar se neeche, ek-ek karke** dabao.

Cell chalne ke baad uske left me **✅ (green tick)** aana chahiye.

---


## Step 1 — ZIP upload karo

**Choose Files** aayega → apni `ajay-ai-assistant.zip` chuno.

> ⏳ Upload me 10-30 sec lagenge. Beech me page band mat karna.
> Agar upload button na dikhe → **Runtime → Restart** karke dobara chalao.


In [ ]:
# --- self-healing helper: har cell akele chal sakta hai ---------------
import os, sys, json, glob

def find_project():
    """Project folder dhoondo, chahe pichhla cell dobara na chala ho."""
    for base in ('/content/work', '/content'):
        if not os.path.isdir(base):
            continue
        for root, dirs, files in os.walk(base):
            if 'backend' in dirs and 'README.md' in files:
                if os.path.isfile(os.path.join(root, 'backend', 'requirements.txt')):
                    return root
    return None

from google.colab import files
import shutil, zipfile

print('📁 ZIP file select karo...\n')
try:
    uploaded = files.upload()
except Exception as e:
    print('❌ Upload nahi hua:', e)
    print('   Fix: Runtime -> Restart runtime, phir ye cell dobara chalao.')
    uploaded = {}

zips = [n for n in uploaded if n.lower().endswith('.zip')]

if not zips:
    print('\n❌ Koi .zip file nahi mili.')
    print('   Aapne jo chuna:', list(uploaded) or '(kuch nahi)')
    print('   Fix: sirf ajay-ai-assistant.zip select karo.')
else:
    zip_name = zips[0]
    size_kb = len(uploaded[zip_name]) // 1024
    print(f'\n✅ Mil gayi: {zip_name}  ({size_kb} KB)')
    if size_kb < 100:
        print('⚠️  File chhoti lag rahi hai — poori download hui thi?')
    print('\n👉 Ab Step 2 wala cell chalao.')


## Step 2 — Extract karo


In [ ]:
# --- self-healing helper: har cell akele chal sakta hai ---------------
import os, sys, json, glob

def find_project():
    """Project folder dhoondo, chahe pichhla cell dobara na chala ho."""
    for base in ('/content/work', '/content'):
        if not os.path.isdir(base):
            continue
        for root, dirs, files in os.walk(base):
            if 'backend' in dirs and 'README.md' in files:
                if os.path.isfile(os.path.join(root, 'backend', 'requirements.txt')):
                    return root
    return None

import zipfile, shutil

# ZIP dhoondo — chahe Step 1 ka variable ab maujood na ho
zip_files = sorted(glob.glob('/content/*.zip'), key=os.path.getmtime, reverse=True)

if not zip_files:
    print('❌ /content me koi ZIP nahi mili.')
    print('   Fix: Step 1 dobara chalao (upload karo).')
else:
    zip_path = zip_files[0]
    print(f'📦 Extract kar raha hoon: {os.path.basename(zip_path)}')

    shutil.rmtree('/content/work', ignore_errors=True)
    os.makedirs('/content/work', exist_ok=True)
    try:
        with zipfile.ZipFile(zip_path) as z:
            z.extractall('/content/work')
    except zipfile.BadZipFile:
        print('❌ ZIP kharab hai (poori download nahi hui).')
        print('   Fix: dobara download karke Step 1 se shuru karo.')

    PROJECT = find_project()
    if not PROJECT:
        print('\n❌ Project folder nahi mila.')
        print('   ZIP ke andar ye tha:')
        for p in sorted(os.listdir('/content/work'))[:10]:
            print('     -', p)
        print('   Fix: sahi ZIP (ajay-ai-assistant.zip) upload karo.')
    else:
        n = sum(len(f) for _, _, f in os.walk(PROJECT))
        print(f'\n✅ Extract ho gaya')
        print(f'   Folder : {PROJECT}')
        print(f'   Files  : {n}')
        ok = os.path.isfile(os.path.join(PROJECT, 'render.yaml'))
        print(f'   render.yaml : {"✅ hai" if ok else "❌ nahi hai"}')
        print('\n👉 Ab Step 3 chalao.')


## Step 3 — Test karo *(optional)*

Ye check karta hai ki code sahi hai. **1-2 min** lagenge.

> ⚠️ Isme bahut sara **gulabi text** aayega — wo normal hai, error nahi.
>
> Ye cell **skip bhi kar sakte ho** — deploy ke liye zaroori nahi hai.
> Agar yahan kuch fail ho jaye tab bhi Step 4/5 chal sakte hain.


In [ ]:
# --- self-healing helper: har cell akele chal sakta hai ---------------
import os, sys, json, glob

def find_project():
    """Project folder dhoondo, chahe pichhla cell dobara na chala ho."""
    for base in ('/content/work', '/content'):
        if not os.path.isdir(base):
            continue
        for root, dirs, files in os.walk(base):
            if 'backend' in dirs and 'README.md' in files:
                if os.path.isfile(os.path.join(root, 'backend', 'requirements.txt')):
                    return root
    return None

import subprocess

PROJECT = find_project()
if not PROJECT:
    print('❌ Project nahi mila — pehle Step 2 chalao.')
else:
    backend = os.path.join(PROJECT, 'backend')
    print('📦 Dependencies install kar raha hoon (1-2 min)...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    'requirements.txt'], cwd=backend,
                   capture_output=True, text=True)

    print('🧪 Tests chala raha hoon...\n')
    p = subprocess.run([sys.executable, '-m', 'pytest', '-q',
                        '--no-header', '-x', '--timeout=60'],
                       cwd=backend, capture_output=True, text=True)
    tail = [l for l in (p.stdout or '').strip().splitlines() if l.strip()]
    for line in tail[-4:]:
        print('  ', line)

    if p.returncode == 0:
        print('\n✅ Sab tests pass — code bilkul sahi hai.')
    else:
        print('\n⚠️  Kuch tests fail hue.')
        print('   Aksar wajah: Colab me internet slow (weather/news tests).')
        print('   Ye deploy rok nahi raha — Step 4 pe aage badh sakte ho.')
    print('\n👉 Ab Step 4 chalao.')


## Step 4 — GitHub details do

Ye cell **3 cheezein** poochhega. Har ek type karke **Enter** dabana:

| Poochhega | Aap kya likho |
|---|---|
| GitHub username | `ajaykumar` *(aapka asli username)* |
| Repo ka naam | khaali chhod do → `ajay-ai-assistant` ho jayega |
| GitHub token | `ghp_...` paste karo *(screen pe dikhega nahi — ye normal hai)* |

> ### 🔑 Token kahan se laayein?
> 1. [github.com/settings/tokens](https://github.com/settings/tokens)
> 2. **Generate new token** → **Generate new token (classic)**
> 3. Note me kuch bhi likho
> 4. **`repo`** wala checkbox **tick karo** ✅ ← *sabse zaroori step*
> 5. Neeche **Generate token** → `ghp_` wala code **copy karo**

> ⚠️ **Phone pe paste karne ka tareeka:** input box pe **der tak dabao** →
> **Paste** aayega. Type karne ki koshish mat karna, galti ho jayegi.

> ⚠️ Token type karte waqt **kuch dikhai nahi dega** (security ke liye).
> Ye normal hai — bas paste karke **Enter** dabao.


In [ ]:
# --- self-healing helper: har cell akele chal sakta hai ---------------
import os, sys, json, glob

def find_project():
    """Project folder dhoondo, chahe pichhla cell dobara na chala ho."""
    for base in ('/content/work', '/content'):
        if not os.path.isdir(base):
            continue
        for root, dirs, files in os.walk(base):
            if 'backend' in dirs and 'README.md' in files:
                if os.path.isfile(os.path.join(root, 'backend', 'requirements.txt')):
                    return root
    return None

from getpass import getpass

print('Teen cheezein poochhunga. Har ek ke baad Enter dabao.\n')

GH_USER = input('1/3  GitHub username : ').strip()
GH_REPO = input('2/3  Repo naam [Enter = ajay-ai-assistant] : ').strip()
GH_REPO = GH_REPO or 'ajay-ai-assistant'
GH_TOKEN = getpass('3/3  Token (dikhega nahi, paste karke Enter) : ').strip()

# saaf-safai: log me se accidental spaces/quotes hatao
GH_USER = GH_USER.strip(' \'"')
GH_TOKEN = GH_TOKEN.strip(' \'"')

print()
problems = []
if not GH_USER:
    problems.append('Username khaali hai')
if ' ' in GH_USER:
    problems.append('Username me space nahi ho sakta')
if not GH_TOKEN:
    problems.append('Token khaali hai — paste hua tha?')
elif len(GH_TOKEN) < 20:
    problems.append(f'Token bahut chhota hai ({len(GH_TOKEN)} chars) — poora paste karo')

if problems:
    print('❌ Ye theek karo:')
    for p in problems:
        print('   -', p)
    print('\n   Fix: ye cell (Step 4) dobara chalao.')
else:
    kind = ('classic' if GH_TOKEN.startswith('ghp_')
            else 'fine-grained' if GH_TOKEN.startswith('github_pat_')
            else 'unknown')
    print(f'✅ Mil gaya')
    print(f'   User  : {GH_USER}')
    print(f'   Repo  : {GH_REPO}')
    print(f'   Token : {GH_TOKEN[:7]}... ({len(GH_TOKEN)} chars, {kind})')
    if kind == 'unknown':
        print('   ⚠️  Token ghp_ ya github_pat_ se shuru hona chahiye.')
        print('      Shayad galat cheez paste ho gayi.')
    print('\n👉 Ab Step 5 chalao.')


## Step 5 — Repo banao aur code push karo

Ye cell sab kuch khud kar dega. **30-60 sec** lagenge.


In [ ]:
# --- self-healing helper: har cell akele chal sakta hai ---------------
import os, sys, json, glob

def find_project():
    """Project folder dhoondo, chahe pichhla cell dobara na chala ho."""
    for base in ('/content/work', '/content'):
        if not os.path.isdir(base):
            continue
        for root, dirs, files in os.walk(base):
            if 'backend' in dirs and 'README.md' in files:
                if os.path.isfile(os.path.join(root, 'backend', 'requirements.txt')):
                    return root
    return None

import subprocess, requests

PROJECT = find_project()

if not PROJECT:
    print('❌ Project nahi mila — Step 2 dobara chalao.')
elif 'GH_TOKEN' not in dir() or not GH_TOKEN:
    print('❌ Token nahi mila — Step 4 dobara chalao.')
else:
    H = {'Authorization': f'token {GH_TOKEN}',
         'Accept': 'application/vnd.github+json'}

    # ---------- 1. token sahi hai? ----------
    print('🔐 Token check kar raha hoon...')
    r = requests.get('https://api.github.com/user', headers=H, timeout=30)

    if r.status_code == 401:
        print('\n❌ Token galat hai (401 Bad credentials).')
        print('   Wajah ho sakti hai:')
        print('     - Token poora paste nahi hua')
        print('     - Token expire ho gaya')
        print('     - Galti se password paste kar diya')
        print('   Fix: naya token banao, phir Step 4 dobara chalao.')
    elif r.status_code != 200:
        print(f'\n❌ GitHub ne mana kiya: {r.status_code}')
        print('  ', r.text[:200])
    else:
        login = r.json()['login']
        scopes = r.headers.get('x-oauth-scopes', '')
        print(f'✅ Login: {login}')

        # ---------- 2. scope check (sabse aam galti) ----------
        classic = GH_TOKEN.startswith('ghp_')
        if classic and 'repo' not in scopes:
            print(f'\n❌ Token me "repo" permission nahi hai.')
            print(f'   Abhi ye hai: {scopes or "(kuch nahi)"}')
            print('   Fix: naya token banao aur **repo** checkbox tick karo:')
            print('        github.com/settings/tokens')
        else:
            if login.lower() != GH_USER.lower():
                print(f'ℹ️  Note: token {login} ka hai, aapne {GH_USER} likha tha.')
                print(f'   {login} use kar raha hoon.')
                GH_USER = login

            # ---------- 3. repo banao ----------
            cr = requests.post('https://api.github.com/user/repos', headers=H,
                               json={'name': GH_REPO, 'private': False,
                                     'description': 'Ajay AI Assistant'},
                               timeout=30)
            if cr.status_code == 201:
                print(f'✅ Repo bana: {GH_REPO}')
            elif cr.status_code == 422:
                print(f'ℹ️  Repo pehle se hai: {GH_REPO} — usi me daal raha hoon')
            elif cr.status_code == 403:
                print('⚠️  403 — token me repo banane ki permission nahi.')
                print('   Agar repo pehle se bana hai to push phir bhi ho sakta hai.')
            else:
                print(f'⚠️  Repo API: {cr.status_code} {cr.text[:150]}')

            # ---------- 4. git push ----------
            print('\n📤 Code bhej raha hoon...')

            def sh(cmd, quiet=False):
                p = subprocess.run(cmd, shell=True, cwd=PROJECT,
                                   capture_output=True, text=True)
                out = (p.stdout + p.stderr).replace(GH_TOKEN, '***')
                return p.returncode, out.strip()

            url = f'https://{GH_TOKEN}@github.com/{GH_USER}/{GH_REPO}.git'
            steps = [
                'git init -q',
                'git config user.email colab@example.com',
                'git config user.name Colab',
                'git add -A',
                'git commit -q -m "Ajay AI Assistant"',
                'git branch -M main',
                'git remote remove origin',
                f'git remote add origin {url}',
                'git push -u origin main --force',
            ]
            fail = None
            for cmd in steps:
                rc, out = sh(cmd)
                soft = ('remote remove' in cmd or 'commit' in cmd)
                if rc != 0 and not soft:
                    fail = (cmd, out)
                    break

            if not fail:
                print('\n🎉🎉 HO GAYA! Code GitHub pe pahunch gaya.\n')
                print(f'   👉 https://github.com/{GH_USER}/{GH_REPO}')
                print('\n   Ab render.com pe jao (neeche Step 6 padho).')
            else:
                cmd, out = fail
                print(f'\n❌ Yahan atka: {cmd.split()[1]}')
                low = out.lower()
                if 'authentication failed' in low or '403' in low:
                    print('   Wajah: token me "repo" permission nahi hai.')
                    print('   Fix: naya token banao, repo tick karo, Step 4 se dobara.')
                elif 'not found' in low:
                    print(f'   Wajah: {GH_USER}/{GH_REPO} nahi mila.')
                    print('   Fix: username spelling check karo.')
                elif 'nothing to commit' in low:
                    print('   Wajah: files nahi mili. Step 2 dobara chalao.')
                else:
                    print('   GitHub ne kaha:')
                    for line in out.splitlines()[-6:]:
                        print('    ', line)


---

# 🩺 Kuch bhi RED aaye to YE cell chalao

Ye khud jaanch karke batayega ki kya galat hai.


In [ ]:
# --- self-healing helper: har cell akele chal sakta hai ---------------
import os, sys, json, glob

def find_project():
    """Project folder dhoondo, chahe pichhla cell dobara na chala ho."""
    for base in ('/content/work', '/content'):
        if not os.path.isdir(base):
            continue
        for root, dirs, files in os.walk(base):
            if 'backend' in dirs and 'README.md' in files:
                if os.path.isfile(os.path.join(root, 'backend', 'requirements.txt')):
                    return root
    return None

import subprocess, platform

print('=' * 46)
print('  🩺 DIAGNOSTIC REPORT')
print('=' * 46)

print(f'\nPython   : {platform.python_version()}')

z = sorted(glob.glob('/content/*.zip'))
print(f'ZIP files: {[os.path.basename(x) for x in z] or "❌ koi nahi"}')
for x in z:
    print(f'           {os.path.basename(x)} = {os.path.getsize(x)//1024} KB')

P = find_project()
print(f'Project  : {P or "❌ nahi mila"}')
if P:
    n = sum(len(f) for _, _, f in os.walk(P))
    print(f'Files    : {n}  (~100+ honi chahiye)')
    for f in ['render.yaml', 'backend/requirements.txt', 'backend/app/main.py']:
        mark = '✅' if os.path.exists(os.path.join(P, f)) else '❌'
        print(f'  {mark} {f}')

have_user = 'GH_USER' in dir() and bool(globals().get('GH_USER'))
have_tok  = 'GH_TOKEN' in dir() and bool(globals().get('GH_TOKEN'))
print(f'\nUsername set : {"✅" if have_user else "❌ Step 4 chalao"}')
print(f'Token set    : {"✅" if have_tok else "❌ Step 4 chalao"}')

if have_tok:
    import requests
    t = globals()['GH_TOKEN']
    print(f'Token format : {t[:7]}... ({len(t)} chars)')
    try:
        r = requests.get('https://api.github.com/user',
                         headers={'Authorization': f'token {t}'}, timeout=20)
        if r.status_code == 200:
            sc = r.headers.get('x-oauth-scopes', '(none)')
            print(f'Token valid  : ✅ {r.json()["login"]}')
            print(f'Permissions  : {sc}')
            if t.startswith('ghp_') and 'repo' not in sc:
                print('   ❌ PROBLEM: "repo" permission missing!')
                print('      Naya token banao, repo checkbox tick karo.')
        else:
            print(f'Token valid  : ❌ HTTP {r.status_code} — token galat/expired')
    except Exception as e:
        print(f'Token check  : ⚠️ {e}')

try:
    g = subprocess.run('git --version', shell=True, capture_output=True, text=True)
    print(f'\ngit          : {g.stdout.strip() or "❌"}')
except Exception:
    print('\ngit          : ❌')

print('\n' + '=' * 46)
print('  Upar ka poora text copy karke chat me bhejo')
print('=' * 46)


---

# 🚀 Step 6 — Render pe live karo

**Notebook ka kaam khatam.** Ab phone ke browser me:

1. [**render.com**](https://render.com) → **Get Started**
2. **GitHub** se sign in *(Google se mat karna)*
3. **Authorize Render**
4. **New +** → **Blueprint**
5. Apna repo chuno → **Connect**
6. **Apply** dabao
7. 3-5 min ruko → **"Your service is live 🎉"**

URL milega:
```
https://ajay-ai-assistant-api.onrender.com
```

**Wo URL phone Chrome me kholo — app chal padegi!** 🎉

Home screen pe icon: Chrome **⋮** → **Add to Home screen**

---

### 🔒 Aakhir me
Token delete kar dena: [github.com/settings/tokens](https://github.com/settings/tokens)
